<a href="https://colab.research.google.com/github/un1u3/ml-labs/blob/main/fusemachines-2026/phase3/NEU_Surface_Defect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement 
You are a junior ML engineer at
SmartForge Manufacturing. The production line images thousands of steel strips
every day. Manual visual inspection is slow, inconsistent, and misses subtle
defects that cause costly downstream failures. The engineering team needs two
things:
1. a model that can identify which type of defect is present, and 
2. evidence that the model is robust enough to handle real production variability
(different lighting, orientations, and surface conditions). Your job is to
build and harden that model using PyTorch.

### Setup 

In [113]:
# imports
import torch 
import torch.nn as nn 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np 

In [114]:
tfm = transforms.Compose([
    transforms.Resize((200,200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])
train_ds = datasets.ImageFolder('NEU-DET/train/images',transform=tfm)
test_ds = datasets.ImageFolder('NEU-DET/validation/images',transform=tfm)


In [115]:
print("Number of images:", len(train_ds))
print("Classes:", train_ds.classes)
print("Class to index:", train_ds.class_to_idx)

Number of images: 1440
Classes: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
Class to index: {'crazing': 0, 'inclusion': 1, 'patches': 2, 'pitted_surface': 3, 'rolled-in_scale': 4, 'scratches': 5}


In [116]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True
)

In [117]:
train_loader

# Part 0 : NN Foundations 

### 1. Implement a simple 2-layer neural network from scratch
using nn.Module no nn.Sequential shortcuts. Define __init__ and forward
explicitly.

In [118]:
class Simple2LayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size ):
        super().__init__()

        #layer 1 
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, X):
        X = X.view(X.size(0), -1) 
        X = self.fc1(X)
        X = self.relu(X)
        X = self.fc2(X)
        return X

In [119]:
model = Simple2LayerNN(3*200*200,128,output_size=len(train_ds.classes))

In [120]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [121]:
for epoch in range(20):
    running_loss = 0.0
    model.train()
    for image, label in train_loader:
        output = model(image)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/20, Loss: {avg_loss}")

Epoch 1/20, Loss: 15.739595376120674
Epoch 2/20, Loss: 7.330330832799276
Epoch 3/20, Loss: 6.441126097573174
Epoch 4/20, Loss: 4.2014437754948935
Epoch 5/20, Loss: 4.391946784655253
Epoch 6/20, Loss: 3.563160648610857
Epoch 7/20, Loss: 2.275518423981137
Epoch 8/20, Loss: 1.9869647022750643
Epoch 9/20, Loss: 3.0215380946795145
Epoch 10/20, Loss: 2.3269922706815933
Epoch 11/20, Loss: 1.5874990079138014
Epoch 12/20, Loss: 1.879078631930881
Epoch 13/20, Loss: 1.737530454662111
Epoch 14/20, Loss: 0.9010066390037537
Epoch 15/20, Loss: 1.0598055015007655
Epoch 16/20, Loss: 0.8771994610627493
Epoch 17/20, Loss: 1.02911181251208
Epoch 18/20, Loss: 0.7181371370951335
Epoch 19/20, Loss: 0.5543680770529641
Epoch 20/20, Loss: 0.6141542525755035


In [122]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 91.53%


### 2. Activation functions: swap ReLU for Sigmoid in your hidden layer and compare convergence over 20 epochs. What do you notice?

In [ ]:
class Simple2LayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size ):
        super().__init__()

        #layer 1 
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()


    def forward(self, X):
        X = X.view(X.size(0), -1) 
        X = self.fc1(X)
        X = self.sigmoid(X)
        X = self.fc2(X)
        return X

In [124]:
model = Simple2LayerNN(3*200*200,128,output_size=len(train_ds.classes))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(20):
    running_loss = 0.0
    model.train()
    for image, label in train_loader:
        output = model(image)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/20, Loss: {avg_loss}")

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Epoch 1/20, Loss: 1.6268503056632149
Epoch 2/20, Loss: 1.5300329287846883
Epoch 3/20, Loss: 1.4998780462476942
Epoch 4/20, Loss: 1.4698729064729479
Epoch 5/20, Loss: 1.4442056788338555
Epoch 6/20, Loss: 1.4196609788470798
Epoch 7/20, Loss: 1.4016477558347913
Epoch 8/20, Loss: 1.3793118317921957
Epoch 9/20, Loss: 1.3536874930063882
Epoch 10/20, Loss: 1.3511026938756308
Epoch 11/20, Loss: 1.3429352680842082
Epoch 12/20, Loss: 1.3404526419109768
Epoch 13/20, Loss: 1.3320120228661432
Epoch 14/20, Loss: 1.3087953011194864
Epoch 15/20, Loss: 1.319506589571635
Epoch 16/20, Loss: 1.2895455360412598
Epoch 17/20, Loss: 1.2756629652447171
Epoch 18/20, Loss: 1.2601603031158448
Epoch 19/20, Loss: 1.2671435038248697
Epoch 20/20, Loss: 1.282940509584215
Test Accuracy: 51.88%


Reflections: after swaping relu with sigmoid over 20 epochs both I see a drop in accuracy from   `49.72%` to `16.67%`, the convergence is slower the training rate loss drecreae at slower rate,This is because the Sigmoid activation function can suffer from the vanishing gradient problem, making it harder for the network to update its weights effectively. In contrast, ReLU allows gradients to flow more easily, resulting in faster convergence and generally better performance for this image classification task.